# **MODELO - TRABAJO FINAL**
**Apellido y Nombres:** Vasquez Caiza Juan Marcelo.

**Carrera:** Ingenieria de Sistemas.

In [192]:
#Librerias basicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#Librerias para Pytorch
import torch
import torch.nn as nn
import torch.optim as optim

In [193]:
data = pd.read_csv('dataset.csv', delimiter=',')
print('INFORMACION DE TIPO DE DATOS')
data.info()
print('\nDATOS VACIOS')
print(pd.isnull(data).sum())

INFORMACION DE TIPO DE DATOS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360000 entries, 0 to 359999
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Fecha             360000 non-null  object 
 1   Producto          360000 non-null  object 
 2   Categoría         360000 non-null  object 
 3   Precio            360000 non-null  float64
 4   UnidadesVendidas  360000 non-null  int64  
 5   Descuento         360000 non-null  int64  
 6   Mes               360000 non-null  int64  
 7   Día_Semana        360000 non-null  int64  
dtypes: float64(1), int64(4), object(3)
memory usage: 22.0+ MB

DATOS VACIOS
Fecha               0
Producto            0
Categoría           0
Precio              0
UnidadesVendidas    0
Descuento           0
Mes                 0
Día_Semana          0
dtype: int64


In [194]:
print(data)

             Fecha                   Producto Categoría  Precio  \
0       21/01/2023           Leche Pil UHT 1L   Lácteos    7.25   
1       04/01/2023        Leche Pil Entera 1L   Lácteos    8.58   
2       01/01/2023       Mantequilla Pil 200g  Derivado   12.01   
3       24/01/2023           Leche Pil UHT 1L   Lácteos    7.40   
4       09/01/2023         Leche Pil Light 1L   Lácteos    8.84   
...            ...                        ...       ...     ...   
359995  23/12/2023  Leche Pil Deslactosada 1L   Lácteos    9.24   
359996  09/12/2023    Leche Pil En Polvo 500g   Lácteos   24.07   
359997  05/12/2023        Leche Pil Entera 1L   Lácteos    8.77   
359998  14/12/2023   Yogurt Bebible Pil 200ml  Derivado    4.45   
359999  01/12/2023        Leche Pil Entera 1L   Lácteos    9.06   

        UnidadesVendidas  Descuento  Mes  Día_Semana  
0                    115          0    1           6  
1                    127          5    1           3  
2                     19      

In [195]:
data['Fecha'] = pd.to_datetime(data['Fecha'], dayfirst=True)

data = pd.get_dummies(data, columns=['Categoría'], drop_first=True)

print(data)

            Fecha                   Producto  Precio  UnidadesVendidas  \
0      2023-01-21           Leche Pil UHT 1L    7.25               115   
1      2023-01-04        Leche Pil Entera 1L    8.58               127   
2      2023-01-01       Mantequilla Pil 200g   12.01                19   
3      2023-01-24           Leche Pil UHT 1L    7.40               113   
4      2023-01-09         Leche Pil Light 1L    8.84                78   
...           ...                        ...     ...               ...   
359995 2023-12-23  Leche Pil Deslactosada 1L    9.24                88   
359996 2023-12-09    Leche Pil En Polvo 500g   24.07                25   
359997 2023-12-05        Leche Pil Entera 1L    8.77               119   
359998 2023-12-14   Yogurt Bebible Pil 200ml    4.45                87   
359999 2023-12-01        Leche Pil Entera 1L    9.06               110   

        Descuento  Mes  Día_Semana  Categoría_Derivado  Categoría_Lácteos  
0               0    1           6 

In [196]:
X = data.drop(columns=['Fecha', 'Producto', 'UnidadesVendidas'])
y = data['UnidadesVendidas']

# **Division de datos**

In [197]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# **Estandarizacion de datos**

In [198]:
from sklearn.preprocessing import StandardScaler

#Estandarizacion de X
scaler_X = StandardScaler()

#ajustar solo en X_train
X_train_scaled = scaler_X.fit_transform(X_train)

#transformar X_test (SIN re-ajustar)
X_test_scaled = scaler_X.transform(X_test)



#Estandarizacion de Y
scaler_y = StandardScaler()

# 1. Ajustar SOLO en y_train. 
# (Necesita .values.reshape(-1, 1) porque es una Serie de Pandas)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))

# 2. Transformar y_test (SIN re-ajustar)
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

In [199]:
X_train_scaled.shape[1]

6

# **Tensores**

In [200]:
torch.cuda.is_available()

True

In [201]:
#Entrenamiento
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).cuda()
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32).view(-1, 1).cuda()

#Test
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).cuda()
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32).view(-1, 1).cuda()

# **Modelo MLP**

In [202]:
class MLP(nn.Module):
    #constructor
    def __init__(self, D_in, H1, H2, D_out):

        #llamamos al constructor de la clase madre
        super(MLP, self).__init__()

        #definicion de capas
        self.hidden1 = nn.Linear(D_in, H1)    #capa oculta 1
        self.hidden2 = nn.Linear(H1, H2)      #capa oculta 2
        self.output = nn.Linear(H2, D_out)    #capa de salida
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        x = self.output(x)
        return x

Inicializacion del modelo

In [203]:
#iniciamos el modelo
model = MLP(6, 64, 32, 1)

#codigo para saber si el modelo esta votando los datos en las cantidades correctas
x_prueba = torch.randn(500, 6)
print(x_prueba)
outputs = model(x_prueba)
outputs.shape

tensor([[-0.6783, -1.3843,  0.4446, -1.0113, -0.6508, -1.0396],
        [ 0.9988,  0.3561,  1.3486,  1.1346,  0.4145, -1.2279],
        [-1.6739,  0.2915,  0.8034, -0.9917, -1.0542, -0.5850],
        ...,
        [ 0.2039, -0.8013, -0.2569,  1.0298,  1.3019,  0.4439],
        [-0.0823, -2.0817, -1.2630,  2.4377, -0.1738, -0.4431],
        [ 1.3018,  0.4522, -0.3440, -0.4165,  0.7020, -0.8470]])


torch.Size([500, 1])

Definir la funcion de **perdida** y el **optimizador**.

In [204]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# **Entrenamiento**

In [205]:
model.to("cuda")
epochs = 10000
log_each = 10
l = []
model.train()

for e in range(1, epochs+1):

    #forward
    y_pred = model(X_train_tensor)

    #loss
    loss = criterion(y_pred, y_train_tensor)
    l.append(loss.item())

    #ponemos a cero los gradientes
    optimizer.zero_grad()

    #backprop calculamos todos los gradientes automaticamente
    loss.backward()

    #update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

Epoch 10/10000 Loss 0.95639
Epoch 20/10000 Loss 0.85109
Epoch 30/10000 Loss 0.76060
Epoch 40/10000 Loss 0.68919
Epoch 50/10000 Loss 0.63783
Epoch 60/10000 Loss 0.59992
Epoch 70/10000 Loss 0.57047
Epoch 80/10000 Loss 0.54687
Epoch 90/10000 Loss 0.52729
Epoch 100/10000 Loss 0.51059
Epoch 110/10000 Loss 0.49602
Epoch 120/10000 Loss 0.48312
Epoch 130/10000 Loss 0.47154
Epoch 140/10000 Loss 0.46104
Epoch 150/10000 Loss 0.45139
Epoch 160/10000 Loss 0.44246
Epoch 170/10000 Loss 0.43410
Epoch 180/10000 Loss 0.42618
Epoch 190/10000 Loss 0.41861
Epoch 200/10000 Loss 0.41133
Epoch 210/10000 Loss 0.40429
Epoch 220/10000 Loss 0.39746
Epoch 230/10000 Loss 0.39084
Epoch 240/10000 Loss 0.38443
Epoch 250/10000 Loss 0.37821
Epoch 260/10000 Loss 0.37219
Epoch 270/10000 Loss 0.36636
Epoch 280/10000 Loss 0.36072
Epoch 290/10000 Loss 0.35526
Epoch 300/10000 Loss 0.34999
Epoch 310/10000 Loss 0.34490
Epoch 320/10000 Loss 0.33998
Epoch 330/10000 Loss 0.33524
Epoch 340/10000 Loss 0.33067
Epoch 350/10000 Loss 0.

# **Evaluacion**

In [206]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_tensor) #Predicciones escaladas

In [207]:
# Revertir predicciones
y_pred_reales = scaler_y.inverse_transform(y_pred_scaled.cpu().numpy())

# Revertir valores reales de prueba (para comparar)
y_test_reales = scaler_y.inverse_transform(y_test_scaled) # o puedes usar y_test.values

In [208]:
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(y_test_reales, y_pred_reales))
print(f"El error promedio del modelo es de +/- {rmse:.2f} unidades.")

El error promedio del modelo es de +/- 10.24 unidades.
